# Braille A-Z Candidate Training

## tl;dr

Notebook ini melatih dan mengekspor kandidat classifier satu sel Braille A-Z. Model seleksi memakai split development untuk menentukan epoch terbaik, kemudian kandidat final dilatih ulang dari awal memakai seluruh gambar unik. Dataset Kaggle hanya memiliki satu gambar sumber per huruf dan banyak variasi augmentasi. Jika split independen berdasarkan sumber tidak mungkin dilakukan, notebook memakai augmentation diagnostic split yang hanya berguna untuk mengontrol proses training dan tidak boleh dianggap sebagai validasi. Notebook tidak mengklaim kesiapan produksi tanpa holdout fisik independen, negative corpus, golden parity, dan benchmark aplikasi pada perangkat nyata.

Output disimpan di /kaggle/working/braille_training_outputs pada Kaggle, atau di BRAILLE_WORKING_DIR saat dijalankan lokal. Notebook tidak mengganti model di aplikasi secara otomatis.

## Context & Methods

Tujuan eksperimen adalah klasifikasi satu crop sel Braille yang sudah dilokalisasi, bukan pembacaan kata, baris, atau halaman.

### Key Assumptions

- Input training: RGB, NHWC, float32, 28 x 28 x 3, rentang 0..1.
- Resize memakai bilinear agar konsisten dengan filtered scaling pada runtime Android.
- Urutan kelas wajib a sampai z dan disimpan sebagai artefak.
- Split berdasarkan source family dipakai jika tersedia; bila tidak, fallback 80/20 diberi label augmentation diagnostic dan hanya dipakai untuk early stopping.
- Holdout produksi harus berasal dari Braille timbul, kamera, material, dan kondisi yang independen dari dataset training.
- Dataset: shanks0465/braille-character-dataset, Kaggle dataset version ID 853989, lisensi CC BY 4.0.
- Lokasi dataset dicari otomatis di /kaggle/input; BRAILLE_DATASET_DIR dapat dipakai sebagai override.

In [ ]:
import os

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

import csv
import hashlib
import json
import platform
import random
import re
import shutil
from collections import Counter
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import PIL
import sklearn
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras import callbacks, layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

random.seed(SEED)
np.random.seed(SEED)
if hasattr(tf.keras.utils, 'set_random_seed'):
    tf.keras.utils.set_random_seed(SEED)
else:
    tf.random.set_seed(SEED)
enable_determinism = getattr(tf.config.experimental, 'enable_op_determinism', None)
if enable_determinism is not None:
    enable_determinism()

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
RAW_DIR_OVERRIDE = os.environ.get('BRAILLE_DATASET_DIR')
RAW_DIR_HINTS = [
    KAGGLE_INPUT_ROOT / 'braille-character-dataset' / 'Braille Dataset' / 'Braille Dataset',
    KAGGLE_INPUT_ROOT / 'braille-character-dataset' / 'Braille Dataset',
    KAGGLE_INPUT_ROOT / 'braille-character-dataset',
]
DEFAULT_WORKING_ROOT = (
    Path('/kaggle/working')
    if Path('/kaggle/working').exists()
    else Path.cwd() / '.braille-training-work'
)
WORKING_ROOT = Path(
    os.environ.get('BRAILLE_WORKING_DIR', str(DEFAULT_WORKING_ROOT))
).expanduser()
WORKING_ROOT.mkdir(parents=True, exist_ok=True)
SPLIT_DIR = WORKING_ROOT / 'braille_dataset_split'
OUTPUT_DIR = WORKING_ROOT / 'braille_training_outputs'
PHYSICAL_HOLDOUT_DIR = Path(os.environ.get(
    'BRAILLE_PHYSICAL_HOLDOUT_DIR',
    '/kaggle/input/braille-physical-holdout/holdout',
)).expanduser()
NEGATIVE_DIR = Path(os.environ.get(
    'BRAILLE_NEGATIVE_DIR',
    '/kaggle/input/braille-physical-holdout/negative',
)).expanduser()
HOLDOUT_MANIFEST_PATH = PHYSICAL_HOLDOUT_DIR.parent / 'holdout_manifest.csv'

EXPECTED_CLASSES = list('abcdefghijklmnopqrstuvwxyz')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
IMG_SIZE = (28, 28)
BATCH_SIZE = 16
TUNING_FRACTION = 0.20
MAX_EPOCHS = 50
RESIZE_INTERPOLATION = 'bilinear'
NEGATIVE_CONFIDENCE_THRESHOLD = 0.90
GOLDEN_PARITY_TOLERANCE = 1e-4

def safe_reset_directory(path, allowed_root):
    resolved_path = path.resolve()
    resolved_root = allowed_root.resolve()
    if resolved_path == resolved_root or resolved_root not in resolved_path.parents:
        raise ValueError(f'Direktori kerja tidak aman: {resolved_path}')
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f'Tipe tidak dapat diserialisasi: {type(value).__name__}')

def save_json(path, payload):
    with open(path, 'w', encoding='utf-8') as target:
        json.dump(payload, target, ensure_ascii=False, indent=2, default=json_default)

safe_reset_directory(OUTPUT_DIR, WORKING_ROOT)
print({
    'python': platform.python_version(),
    'tensorflow': tf.__version__,
    'numpy': np.__version__,
    'scikit_learn': sklearn.__version__,
    'matplotlib': matplotlib.__version__,
    'working_root': str(WORKING_ROOT),
})

## Data

Cell berikut memeriksa kelengkapan A-Z, jumlah file per kelas, exact duplicate, dan source family dari nama file. Exact duplicate dengan label yang sama dideduplicasi berdasarkan SHA-256; konflik label tetap menghentikan proses. Parser source family diuji terhadap indeks augmentasi satu dan dua digit agar satu gambar dasar tidak keliru dianggap sebagai beberapa sumber.

In [ ]:
def discover_braille_directory():
    mounted_inputs = (
        sorted(path for path in KAGGLE_INPUT_ROOT.iterdir() if path.is_dir())
        if KAGGLE_INPUT_ROOT.is_dir()
        else []
    )
    search_roots = []
    if RAW_DIR_OVERRIDE:
        override_path = Path(RAW_DIR_OVERRIDE).expanduser()
        if not override_path.is_dir():
            raise FileNotFoundError(
                f'BRAILLE_DATASET_DIR tidak valid: {override_path}'
            )
        search_roots.append(override_path)
    search_roots.extend(path for path in RAW_DIR_HINTS if path.is_dir())
    search_roots.extend(
        path for path in mounted_inputs if 'braille' in path.name.lower()
    )
    search_roots.extend(mounted_inputs)

    unique_roots = []
    seen_roots = set()
    for path in search_roots:
        resolved = path.resolve()
        if resolved not in seen_roots:
            unique_roots.append(path)
            seen_roots.add(resolved)

    candidates = []
    visited_directories = set()
    for search_root in unique_roots:
        for current_root, child_directories, filenames in os.walk(search_root):
            current_path = Path(current_root)
            resolved_current = current_path.resolve()
            if resolved_current in visited_directories:
                child_directories[:] = []
                continue
            visited_directories.add(resolved_current)
            relative_depth = len(current_path.relative_to(search_root).parts)
            if relative_depth >= 6:
                child_directories[:] = []
            image_names = [
                name for name in filenames
                if Path(name).suffix.lower() in IMAGE_EXTENSIONS
            ]
            if not image_names:
                continue
            label_counts = Counter(
                name[0].lower() for name in image_names
                if name and name[0].lower() in EXPECTED_CLASSES
            )
            all_images_have_known_labels = sum(label_counts.values()) == len(image_names)
            if (
                all_images_have_known_labels
                and all(label_counts[label] > 0 for label in EXPECTED_CLASSES)
            ):
                candidates.append({
                    'path': current_path,
                    'image_count': len(image_names),
                    'minimum_class_count': min(label_counts.values()),
                })

    if not candidates:
        mount_names = [path.name for path in mounted_inputs]
        raise FileNotFoundError(
            'Folder gambar Braille A-Z tidak ditemukan. '
            f'Kaggle inputs yang terpasang: {mount_names or ["tidak ada"]}. '
            'Klik Add Input, tambahkan shanks0465/braille-character-dataset, '
            'atau set environment variable BRAILLE_DATASET_DIR ke folder gambarnya.'
        )

    selected = max(
        candidates,
        key=lambda item: (item['minimum_class_count'], item['image_count']),
    )
    return selected['path'], candidates

RAW_DIR, dataset_candidates = discover_braille_directory()
print({
    'selected_dataset_directory': str(RAW_DIR),
    'candidate_directories': [str(item['path']) for item in dataset_candidates],
})

raw_files = sorted(
    path for path in RAW_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)
if not raw_files:
    raise ValueError('Dataset tidak memiliki file gambar yang didukung.')

augmentation_pattern = re.compile(
    r'^([a-z]\d+\.(?:jpe?g|png))\d+(?:dim|rot|whs)\.(?:jpe?g|png)$',
    re.IGNORECASE,
)
parser_test_names = [
    'a1.JPG0dim.jpg',
    'a1.JPG10dim.jpg',
    'a1.JPG19rot.jpg',
]
parser_test_families = {
    augmentation_pattern.fullmatch(name).group(1).lower()
    for name in parser_test_names
}
if parser_test_families != {'a1.jpg'}:
    raise AssertionError(f'Parser source family salah: {parser_test_families}')
records = []
files_by_label = {label: [] for label in EXPECTED_CLASSES}
unexpected_files = []
unparsed_augmentation_files = []

for path in raw_files:
    label = path.name[0].lower()
    if label not in files_by_label:
        unexpected_files.append(path.name)
        continue
    match = augmentation_pattern.fullmatch(path.name)
    if match is None:
        unparsed_augmentation_files.append(path.name)
        continue
    source_family = match.group(1).lower()
    record = {
        'path': path,
        'filename': path.name,
        'label': label,
        'source_family': source_family,
        'sha256': sha256_file(path),
    }
    records.append(record)
    files_by_label[label].append(record)

missing_classes = [label for label, items in files_by_label.items() if not items]
if missing_classes:
    raise ValueError(f'Kelas tidak lengkap: {missing_classes}')
if unexpected_files:
    raise ValueError(f'Nama file dengan label tidak dikenal: {unexpected_files[:10]}')
if unparsed_augmentation_files:
    raise ValueError(
        'Nama file augmentasi tidak sesuai kontrak: '
        f'{unparsed_augmentation_files[:10]}'
    )

raw_class_counts = {label: len(files_by_label[label]) for label in EXPECTED_CLASSES}
records_by_hash = {}
for item in records:
    records_by_hash.setdefault(item['sha256'], []).append(item)

hash_label_conflicts = []
for digest, duplicate_records in sorted(records_by_hash.items()):
    duplicate_labels = sorted({item['label'] for item in duplicate_records})
    if len(duplicate_labels) > 1:
        hash_label_conflicts.append({
            'sha256': digest,
            'labels': duplicate_labels,
            'filenames': sorted(item['filename'] for item in duplicate_records),
        })

if hash_label_conflicts:
    save_json(OUTPUT_DIR / 'hash_label_conflicts.json', hash_label_conflicts)
    raise ValueError(
        'Gambar identik memiliki label berbeda. Lihat hash_label_conflicts.json.'
    )

unique_records = []
duplicate_groups = []
for digest, duplicate_records in sorted(records_by_hash.items()):
    ordered_records = sorted(duplicate_records, key=lambda item: item['filename'])
    retained_record = ordered_records[0]
    unique_records.append(retained_record)
    if len(ordered_records) > 1:
        duplicate_groups.append({
            'sha256': digest,
            'label': retained_record['label'],
            'retained_filename': retained_record['filename'],
            'discarded_filenames': [
                item['filename'] for item in ordered_records[1:]
            ],
            'source_families': sorted({
                item['source_family'] for item in ordered_records
            }),
        })

raw_image_count = len(records)
records = sorted(unique_records, key=lambda item: (item['label'], item['filename']))
files_by_label = {label: [] for label in EXPECTED_CLASSES}
for item in records:
    files_by_label[item['label']].append(item)

class_counts = {label: len(files_by_label[label]) for label in EXPECTED_CLASSES}
source_family_counts = {
    label: len({item['source_family'] for item in files_by_label[label]})
    for label in EXPECTED_CLASSES
}
exact_duplicate_files = raw_image_count - len(records)
save_json(OUTPUT_DIR / 'exact_duplicate_groups.json', duplicate_groups)

dataset_profile = {
    'dataset_slug': 'shanks0465/braille-character-dataset',
    'resolved_dataset_directory': str(RAW_DIR),
    'kaggle_dataset_version_id': 853989,
    'license': 'CC BY 4.0',
    'raw_image_count': raw_image_count,
    'image_count': len(records),
    'class_count': len(class_counts),
    'raw_class_counts': raw_class_counts,
    'class_counts': class_counts,
    'source_family_counts': source_family_counts,
    'exact_duplicate_files': exact_duplicate_files,
    'exact_duplicate_group_count': len(duplicate_groups),
    'exact_duplicate_rate': exact_duplicate_files / raw_image_count,
    'intended_use': 'development_and_tuning_only',
}
save_json(OUTPUT_DIR / 'dataset_profile.json', dataset_profile)
print(json.dumps(dataset_profile, indent=2))

In [ ]:
safe_reset_directory(SPLIT_DIR, WORKING_ROOT)
train_dir = SPLIT_DIR / 'train'
tune_dir = SPLIT_DIR / 'tune'
full_dir = SPLIT_DIR / 'full'
split_rows = []

minimum_source_family_count = min(source_family_counts.values())
source_family_split_available = minimum_source_family_count >= 2
if source_family_split_available:
    split_strategy = 'source_family'
    development_evaluation_role = 'source_group_tuning_only'
    development_evaluation_name = 'Development source-group tuning'
    development_output_stem = 'development_tuning'
else:
    split_strategy = 'augmentation_diagnostic_fallback'
    development_evaluation_role = 'diagnostic_only'
    development_evaluation_name = 'Augmentation diagnostic'
    development_output_stem = 'augmentation_diagnostic'

for label in EXPECTED_CLASSES:
    label_records = sorted(files_by_label[label], key=lambda item: item['filename'])
    full_destination_dir = full_dir / label
    full_destination_dir.mkdir(parents=True, exist_ok=True)
    for item in label_records:
        shutil.copy2(item['path'], full_destination_dir / item['filename'])
    if source_family_split_available:
        source_families = sorted({item['source_family'] for item in label_records})
        train_family_ids, tune_family_ids = train_test_split(
            source_families,
            test_size=TUNING_FRACTION,
            random_state=SEED,
            shuffle=True,
        )
        partition_by_family = {family: 'train' for family in train_family_ids}
        partition_by_family.update({family: 'tune' for family in tune_family_ids})
        partitioned_records = (
            ('train', [
                item for item in label_records
                if partition_by_family[item['source_family']] == 'train'
            ]),
            ('tune', [
                item for item in label_records
                if partition_by_family[item['source_family']] == 'tune'
            ]),
        )
    else:
        train_records, tune_records = train_test_split(
            label_records,
            test_size=TUNING_FRACTION,
            random_state=SEED,
            shuffle=True,
        )
        partitioned_records = (('train', train_records), ('tune', tune_records))

    for partition, selected_records in partitioned_records:
        destination_root = train_dir if partition == 'train' else tune_dir
        destination_dir = destination_root / label
        destination_dir.mkdir(parents=True, exist_ok=True)
        for item in selected_records:
            destination = destination_dir / item['filename']
            shutil.copy2(item['path'], destination)
            split_rows.append({
                'filename': item['filename'],
                'label': label,
                'partition': partition,
                'source_family': item['source_family'],
                'sha256': item['sha256'],
            })

if len(split_rows) != len(records):
    raise ValueError(
        f'Jumlah hasil split tidak cocok: {len(split_rows)} != {len(records)}'
    )
full_copy_count = sum(
    1 for path in full_dir.rglob('*')
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)
if full_copy_count != len(records):
    raise ValueError(
        f'Jumlah data full refit tidak cocok: {full_copy_count} != {len(records)}'
    )
partition_class_counts = {
    partition: {
        label: sum(
            row['partition'] == partition and row['label'] == label
            for row in split_rows
        )
        for label in EXPECTED_CLASSES
    }
    for partition in ('train', 'tune')
}
missing_partition_classes = [
    f'{partition}:{label}'
    for partition, counts in partition_class_counts.items()
    for label, count in counts.items()
    if count == 0
]
if missing_partition_classes:
    raise ValueError(f'Kelas kosong setelah split: {missing_partition_classes}')

train_hashes = {row['sha256'] for row in split_rows if row['partition'] == 'train'}
tune_hashes = {row['sha256'] for row in split_rows if row['partition'] == 'tune'}
if train_hashes & tune_hashes:
    raise ValueError('Exact duplicate ditemukan di train dan tune.')

train_families = {row['source_family'] for row in split_rows if row['partition'] == 'train'}
tune_families = {row['source_family'] for row in split_rows if row['partition'] == 'tune'}
shared_source_families = sorted(train_families & tune_families)
if source_family_split_available and shared_source_families:
    raise ValueError(f'Source-family leakage ditemukan: {shared_source_families[:10]}')

if source_family_split_available:
    split_caveat = (
        'Source family tidak dibagi lintas train dan tune. Tune tetap hanya untuk '
        'early stopping, bukan holdout produksi.'
    )
else:
    split_caveat = (
        'Dataset hanya memiliki satu source family per kelas. Train dan diagnostic '
        'berbagi sumber dasar; semua metrik diagnostic bersifat optimistis dan tidak '
        'boleh digunakan sebagai bukti generalisasi atau akurasi produk.'
    )

with open(OUTPUT_DIR / 'development_split.csv', 'w', newline='', encoding='utf-8') as target:
    writer = csv.DictWriter(
        target,
        fieldnames=['filename', 'label', 'partition', 'source_family', 'sha256'],
    )
    writer.writeheader()
    writer.writerows(split_rows)

split_profile = {
    'strategy': split_strategy,
    'development_evaluation_role': development_evaluation_role,
    'source_family_independent': source_family_split_available,
    'minimum_source_family_count_per_class': minimum_source_family_count,
    'train_count': sum(row['partition'] == 'train' for row in split_rows),
    'tune_count': sum(row['partition'] == 'tune' for row in split_rows),
    'unique_dataset_count': len(records),
    'final_refit_count': full_copy_count,
    'partition_class_counts': partition_class_counts,
    'shared_source_family_count': len(shared_source_families),
    'train_source_family_count': len(train_families),
    'tune_source_family_count': len(tune_families),
    'shared_source_family_examples': shared_source_families[:10],
    'caveat': split_caveat,
}
save_json(OUTPUT_DIR / 'split_profile.json', split_profile)
print(json.dumps(split_profile, indent=2))

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=5,
    width_shift_range=0.05,
    height_shift_range=0.05,
)
evaluation_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_generator = train_datagen.flow_from_directory(
    str(train_dir),
    target_size=IMG_SIZE,
    color_mode='rgb',
    classes=EXPECTED_CLASSES,
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    interpolation=RESIZE_INTERPOLATION,
)
tune_generator = evaluation_datagen.flow_from_directory(
    str(tune_dir),
    target_size=IMG_SIZE,
    color_mode='rgb',
    classes=EXPECTED_CLASSES,
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False,
    interpolation=RESIZE_INTERPOLATION,
)
full_generator = train_datagen.flow_from_directory(
    str(full_dir),
    target_size=IMG_SIZE,
    color_mode='rgb',
    classes=EXPECTED_CLASSES,
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    interpolation=RESIZE_INTERPOLATION,
)

class_names = [
    name for name, index in sorted(
        train_generator.class_indices.items(),
        key=lambda item: item[1],
    )
]
if class_names != EXPECTED_CLASSES:
    raise ValueError(f'Urutan label salah: {class_names}')
if tune_generator.class_indices != train_generator.class_indices:
    raise ValueError('Mapping label train dan tune berbeda.')
if full_generator.class_indices != train_generator.class_indices:
    raise ValueError('Mapping label full refit dan train berbeda.')
if full_generator.samples != len(records):
    raise ValueError(
        f'Jumlah generator full refit salah: {full_generator.samples} != {len(records)}'
    )

save_json(OUTPUT_DIR / 'labels.json', class_names)
print({'class_indices': train_generator.class_indices})

## Training

Arsitektur dipertahankan agar kompatibel dengan kontrak aplikasi: tiga Conv2D, dua MaxPooling, Dense 128, dan Softmax 26 kelas. Best weights dipulihkan berdasarkan tuning loss dan disimpan terpisah dari model aplikasi.

In [ ]:
def build_compiled_model():
    candidate = models.Sequential([
        layers.Input(shape=(28, 28, 3), name='braille_input'),
        layers.Conv2D(32, (3, 3), activation='relu', name='conv2d_1'),
        layers.MaxPooling2D((2, 2), name='pool_1'),
        layers.Conv2D(64, (3, 3), activation='relu', name='conv2d_2'),
        layers.MaxPooling2D((2, 2), name='pool_2'),
        layers.Conv2D(128, (3, 3), activation='relu', name='conv2d_3'),
        layers.Flatten(name='flatten'),
        layers.Dense(128, activation='relu', name='dense_1'),
        layers.Dropout(0.3, name='dropout'),
        layers.Dense(26, activation='softmax', name='predictions'),
    ], name='braille_cell_classifier')
    candidate.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    if candidate.count_params() != 244186:
        raise ValueError(
            f'Arsitektur berubah: {candidate.count_params()} parameter.'
        )
    return candidate

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
selection_model = build_compiled_model()
selection_model.summary()

In [ ]:
def write_history_csv(history_object, path):
    history_keys = list(history_object.history.keys())
    with open(path, 'w', newline='', encoding='utf-8') as target:
        writer = csv.writer(target)
        writer.writerow(['epoch'] + history_keys)
        for epoch_index in range(len(history_object.history[history_keys[0]])):
            writer.writerow(
                [epoch_index + 1] + [
                    history_object.history[key][epoch_index]
                    for key in history_keys
                ]
            )

selection_checkpoint_path = OUTPUT_DIR / 'selection_best_model.keras'
selection_history = selection_model.fit(
    train_generator,
    validation_data=tune_generator,
    epochs=MAX_EPOCHS,
    callbacks=[
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=8,
            restore_best_weights=True,
            verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            verbose=1,
        ),
        callbacks.ModelCheckpoint(
            str(selection_checkpoint_path),
            monitor='val_loss',
            save_best_only=True,
            verbose=1,
        ),
    ],
)

selection_history_path = OUTPUT_DIR / 'selection_training_history.csv'
write_history_csv(selection_history, selection_history_path)
best_epoch = int(np.argmin(selection_history.history['val_loss'])) + 1
selection_model = tf.keras.models.load_model(str(selection_checkpoint_path))
print({
    'selection_best_epoch': best_epoch,
    'selection_checkpoint': str(selection_checkpoint_path),
})

## Results

Metrik berikut selalu merupakan development metrics, bukan production accuracy. Jika split_profile.json menunjukkan role diagnostic_only, train dan diagnostic berasal dari gambar dasar yang sama; angka tersebut hanya memeriksa proses training dan tidak mengukur generalisasi.

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(selection_history.history['loss'], label='Train')
plt.plot(selection_history.history['val_loss'], label=development_evaluation_name)
plt.title('Loss by epoch')
plt.xlabel('Epoch')
plt.ylabel('Categorical cross-entropy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(selection_history.history['accuracy'], label='Train')
plt.plot(selection_history.history['val_accuracy'], label=development_evaluation_name)
plt.title('Accuracy by epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=160, bbox_inches='tight')
plt.show()

def evaluate_iterator(evaluation_model, iterator, title, output_stem):
    iterator.reset()
    probabilities = evaluation_model.predict(iterator, verbose=0)
    true_labels = np.asarray(iterator.classes)
    predicted_labels = np.argmax(probabilities, axis=1)
    report = classification_report(
        true_labels,
        predicted_labels,
        labels=list(range(len(class_names))),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    matrix = confusion_matrix(
        true_labels,
        predicted_labels,
        labels=list(range(len(class_names))),
    )
    figure, axis = plt.subplots(figsize=(12, 12))
    ConfusionMatrixDisplay(matrix, display_labels=class_names).plot(
        ax=axis,
        cmap='Blues',
        colorbar=False,
        values_format='d',
        xticks_rotation=90,
    )
    axis.set_title(title)
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / f'{output_stem}_confusion_matrix.png', dpi=160)
    plt.show()
    summary = {
        'sample_count': int(len(true_labels)),
        'accuracy': float(report['accuracy']),
        'macro_precision': float(report['macro avg']['precision']),
        'macro_recall': float(report['macro avg']['recall']),
        'macro_f1': float(report['macro avg']['f1-score']),
    }
    save_json(OUTPUT_DIR / f'{output_stem}_classification_report.json', report)
    save_json(OUTPUT_DIR / f'{output_stem}_confusion_matrix.json', matrix.tolist())
    return summary, report

development_summary, development_report = evaluate_iterator(
    selection_model,
    tune_generator,
    f'{development_evaluation_name} confusion matrix',
    development_output_stem,
)
development_summary.update({
    'role': development_evaluation_role,
    'split_strategy': split_strategy,
    'source_family_independent': source_family_split_available,
    'caveat': split_caveat,
})
print(json.dumps(development_summary, indent=2))

selection_learning_rates = selection_history.history.get(
    'learning_rate',
    [0.001] * len(selection_history.history['loss']),
)
final_refit_learning_rates = [
    float(value) for value in selection_learning_rates[:best_epoch]
]

class ReplayLearningRate(callbacks.Callback):
    def __init__(self, learning_rates):
        super().__init__()
        self.learning_rates = learning_rates

    def on_epoch_begin(self, epoch, logs=None):
        learning_rate = self.learning_rates[min(epoch, len(self.learning_rates) - 1)]
        self.model.optimizer.learning_rate.assign(learning_rate)

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
full_generator.reset()
model = build_compiled_model()
final_history = model.fit(
    full_generator,
    epochs=best_epoch,
    callbacks=[ReplayLearningRate(final_refit_learning_rates)],
)
final_history_path = OUTPUT_DIR / 'final_refit_history.csv'
write_history_csv(final_history, final_history_path)
candidate_keras_path = OUTPUT_DIR / 'braille_model_candidate.keras'
model.save(str(candidate_keras_path))
final_refit_summary = {
    'sample_count': int(full_generator.samples),
    'epochs': best_epoch,
    'uses_all_unique_development_images': full_generator.samples == len(records),
    'learning_rate_schedule_replayed_from_selection': True,
    'saved_model': str(candidate_keras_path),
}
print(json.dumps(final_refit_summary, indent=2))

### Independent physical holdout and negative corpus

Jika dataset tambahan tersedia, buat folder holdout/a sampai holdout/z dan folder negative di dataset Kaggle terpisah. Sertakan holdout_manifest.csv dengan ground truth, source ID, perangkat, material, dan kondisi pengambilan. Ketiadaan corpus ini tidak menghentikan training, tetapi tetap memblokir release.

In [ ]:
holdout_summary = None
holdout_report = None
holdout_manifest_present = HOLDOUT_MANIFEST_PATH.is_file()
holdout_manifest_required_columns = {
    'filename', 'label', 'source_id', 'capture_session',
    'device', 'material', 'lighting', 'sha256',
}
holdout_manifest_validation = {
    'path': str(HOLDOUT_MANIFEST_PATH),
    'present': holdout_manifest_present,
    'valid': False,
    'row_count': 0,
    'errors': [],
}
if holdout_manifest_present:
    with open(HOLDOUT_MANIFEST_PATH, newline='', encoding='utf-8-sig') as source:
        reader = csv.DictReader(source)
        manifest_columns = set(reader.fieldnames or [])
        holdout_manifest_rows = list(reader)
    missing_manifest_columns = sorted(
        holdout_manifest_required_columns - manifest_columns
    )
    if missing_manifest_columns:
        holdout_manifest_validation['errors'].append(
            f'Kolom wajib tidak ada: {missing_manifest_columns}'
        )
    if not holdout_manifest_rows:
        holdout_manifest_validation['errors'].append('Manifest kosong.')
    holdout_manifest_validation['row_count'] = len(holdout_manifest_rows)
    if not missing_manifest_columns and holdout_manifest_rows:
        manifest_filenames = [row['filename'] for row in holdout_manifest_rows]
        duplicate_manifest_filenames = sorted(
            name for name, count in Counter(manifest_filenames).items()
            if count > 1
        )
        if duplicate_manifest_filenames:
            holdout_manifest_validation['errors'].append(
                'Filename manifest duplikat: '
                f'{duplicate_manifest_filenames[:10]}'
            )
        invalid_manifest_labels = sorted({
            row['label'] for row in holdout_manifest_rows
            if row['label'] not in EXPECTED_CLASSES
        })
        if invalid_manifest_labels:
            holdout_manifest_validation['errors'].append(
                f'Label manifest tidak valid: {invalid_manifest_labels}'
            )
        missing_manifest_classes = sorted(
            set(EXPECTED_CLASSES) - {row['label'] for row in holdout_manifest_rows}
        )
        if missing_manifest_classes:
            holdout_manifest_validation['errors'].append(
                f'Kelas manifest tidak lengkap: {missing_manifest_classes}'
            )
        if PHYSICAL_HOLDOUT_DIR.is_dir():
            holdout_files = [
                path for path in PHYSICAL_HOLDOUT_DIR.rglob('*')
                if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
            ]
            files_by_name = {}
            for path in holdout_files:
                files_by_name.setdefault(path.name, []).append(path)
            ambiguous_names = sorted(
                name for name, paths in files_by_name.items() if len(paths) > 1
            )
            if ambiguous_names:
                holdout_manifest_validation['errors'].append(
                    f'Nama file holdout ambigu: {ambiguous_names[:10]}'
                )
            missing_manifest_files = sorted(
                set(files_by_name) - set(manifest_filenames)
            )
            missing_holdout_files = sorted(
                set(manifest_filenames) - set(files_by_name)
            )
            if missing_manifest_files:
                holdout_manifest_validation['errors'].append(
                    f'File tanpa baris manifest: {missing_manifest_files[:10]}'
                )
            if missing_holdout_files:
                holdout_manifest_validation['errors'].append(
                    f'Baris manifest tanpa file: {missing_holdout_files[:10]}'
                )
            for row in holdout_manifest_rows:
                paths = files_by_name.get(row['filename'], [])
                if len(paths) == 1 and sha256_file(paths[0]) != row['sha256'].lower():
                    holdout_manifest_validation['errors'].append(
                        f'SHA-256 tidak cocok: {row["filename"]}'
                    )
    holdout_manifest_validation['valid'] = not holdout_manifest_validation['errors']
save_json(
    OUTPUT_DIR / 'holdout_manifest_validation.json',
    holdout_manifest_validation,
)
holdout_manifest_valid = holdout_manifest_validation['valid']

if PHYSICAL_HOLDOUT_DIR.is_dir():
    missing_holdout_classes = [
        label for label in EXPECTED_CLASSES
        if not (PHYSICAL_HOLDOUT_DIR / label).is_dir()
    ]
    if missing_holdout_classes:
        raise ValueError(f'Holdout tidak mencakup kelas: {missing_holdout_classes}')
    holdout_generator = evaluation_datagen.flow_from_directory(
        str(PHYSICAL_HOLDOUT_DIR),
        target_size=IMG_SIZE,
        color_mode='rgb',
        classes=EXPECTED_CLASSES,
        class_mode='categorical',
        batch_size=BATCH_SIZE,
        shuffle=False,
        interpolation=RESIZE_INTERPOLATION,
    )
    holdout_summary, holdout_report = evaluate_iterator(
        model,
        holdout_generator,
        'Independent physical holdout confusion matrix',
        'physical_holdout',
    )
    holdout_summary.update({
        'role': 'independent_physical_holdout',
        'manifest_valid': holdout_manifest_valid,
    })
    print(json.dumps(holdout_summary, indent=2))
else:
    print(f'Physical holdout belum tersedia di {PHYSICAL_HOLDOUT_DIR}.')

negative_summary = None
if NEGATIVE_DIR.is_dir():
    negative_files = sorted(
        path for path in NEGATIVE_DIR.rglob('*')
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )
    if not negative_files:
        raise ValueError('Folder negative tersedia tetapi tidak memiliki gambar.')
    negative_results = []
    for path in negative_files:
        image = tf.keras.preprocessing.image.load_img(
            str(path),
            target_size=IMG_SIZE,
            color_mode='rgb',
            interpolation=RESIZE_INTERPOLATION,
        )
        tensor = tf.keras.preprocessing.image.img_to_array(image).astype(np.float32) / 255.0
        probabilities = model(np.expand_dims(tensor, axis=0), training=False).numpy()[0]
        predicted_index = int(np.argmax(probabilities))
        confidence = float(probabilities[predicted_index])
        negative_results.append({
            'filename': path.name,
            'predicted_label': class_names[predicted_index],
            'confidence': confidence,
            'false_accept': confidence >= NEGATIVE_CONFIDENCE_THRESHOLD,
        })
    false_accepts = [item for item in negative_results if item['false_accept']]
    negative_summary = {
        'sample_count': len(negative_results),
        'confidence_threshold': NEGATIVE_CONFIDENCE_THRESHOLD,
        'false_accept_count': len(false_accepts),
        'false_accept_rate': len(false_accepts) / len(negative_results),
        'highest_confidence': max(item['confidence'] for item in negative_results),
        'false_accept_examples': false_accepts[:50],
    }
    save_json(OUTPUT_DIR / 'negative_guardrail.json', negative_summary)
    print(json.dumps(negative_summary, indent=2))
else:
    print(f'Negative corpus belum tersedia di {NEGATIVE_DIR}.')

### Export and golden parity

TFLite diekspor sebagai kandidat. Satu tuning sample per kelas dipakai untuk membandingkan probabilitas Keras dan TFLite. Ini memeriksa conversion parity, bukan camera/device parity.

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
candidate_tflite_path = OUTPUT_DIR / 'braille_model_candidate.tflite'
with open(candidate_tflite_path, 'wb') as target:
    target.write(tflite_model)

interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]
if list(input_detail['shape']) != [1, 28, 28, 3] or input_detail['dtype'] != np.float32:
    raise ValueError(f'Kontrak input TFLite salah: {input_detail}')
if list(output_detail['shape']) != [1, 26] or output_detail['dtype'] != np.float32:
    raise ValueError(f'Kontrak output TFLite salah: {output_detail}')

tune_classes = np.asarray(tune_generator.classes)
golden_records = []
for class_index, expected_label in enumerate(class_names):
    candidate_positions = np.flatnonzero(tune_classes == class_index)
    if candidate_positions.size == 0:
        raise ValueError(f'Tidak ada golden candidate untuk kelas {expected_label}.')
    path = Path(tune_generator.filepaths[int(candidate_positions[0])])
    image = tf.keras.preprocessing.image.load_img(
        str(path),
        target_size=IMG_SIZE,
        color_mode='rgb',
        interpolation=RESIZE_INTERPOLATION,
    )
    tensor = tf.keras.preprocessing.image.img_to_array(image).astype(np.float32) / 255.0
    batch = np.expand_dims(tensor, axis=0)
    keras_probabilities = model(batch, training=False).numpy()[0]
    interpreter.set_tensor(input_detail['index'], batch)
    interpreter.invoke()
    tflite_probabilities = interpreter.get_tensor(output_detail['index'])[0]
    keras_index = int(np.argmax(keras_probabilities))
    tflite_index = int(np.argmax(tflite_probabilities))
    golden_records.append({
        'filename': path.name,
        'expected_label': expected_label,
        'keras_label': class_names[keras_index],
        'tflite_label': class_names[tflite_index],
        'keras_expected_label_match': keras_index == class_index,
        'tflite_expected_label_match': tflite_index == class_index,
        'top1_match': keras_index == tflite_index,
        'max_absolute_probability_difference': float(
            np.max(np.abs(keras_probabilities - tflite_probabilities))
        ),
    })

golden_top1_match_rate = sum(item['top1_match'] for item in golden_records) / len(golden_records)
golden_max_difference = max(
    item['max_absolute_probability_difference'] for item in golden_records
)
golden_parity_passed = (
    golden_top1_match_rate == 1.0
    and golden_max_difference <= GOLDEN_PARITY_TOLERANCE
)
golden_parity = {
    'purpose': 'keras_tflite_conversion_parity_only',
    'not_an_independent_accuracy_evaluation': True,
    'sample_count': len(golden_records),
    'one_sample_per_class': True,
    'keras_expected_label_match_rate': sum(
        item['keras_expected_label_match'] for item in golden_records
    ) / len(golden_records),
    'tflite_expected_label_match_rate': sum(
        item['tflite_expected_label_match'] for item in golden_records
    ) / len(golden_records),
    'top1_match_rate': golden_top1_match_rate,
    'max_absolute_probability_difference': golden_max_difference,
    'tolerance': GOLDEN_PARITY_TOLERANCE,
    'passed': golden_parity_passed,
    'records': golden_records,
}
save_json(OUTPUT_DIR / 'golden_parity.json', golden_parity)
print(json.dumps({key: value for key, value in golden_parity.items() if key != 'records'}, indent=2))

In [ ]:
environment = {
    'python': platform.python_version(),
    'tensorflow': tf.__version__,
    'numpy': np.__version__,
    'scikit_learn': sklearn.__version__,
    'matplotlib': matplotlib.__version__,
    'pillow': PIL.__version__,
    'keras': tf.keras.__version__,
    'platform': platform.platform(),
    'seed': SEED,
}
preprocessing_contract = {
    'shape': [1, 28, 28, 3],
    'layout': 'NHWC',
    'dtype': 'float32',
    'channel_order': 'RGB',
    'value_range': [0.0, 1.0],
    'normalization': 'pixel / 255.0',
    'resize_interpolation': RESIZE_INTERPOLATION,
}
requirements_lines = [
    f'tensorflow=={tf.__version__}',
    f'numpy=={np.__version__}',
    f'scikit-learn=={sklearn.__version__}',
    f'matplotlib=={matplotlib.__version__}',
    f'pillow=={PIL.__version__}',
]
with open(OUTPUT_DIR / 'training_requirements.txt', 'w', encoding='utf-8') as target:
    target.write('\n'.join(requirements_lines) + '\n')

artifact_names = [
    candidate_keras_path.name,
    candidate_tflite_path.name,
    selection_checkpoint_path.name,
    'labels.json',
    'dataset_profile.json',
    'exact_duplicate_groups.json',
    'development_split.csv',
    'split_profile.json',
    'selection_training_history.csv',
    'final_refit_history.csv',
    'training_curves.png',
    f'{development_output_stem}_classification_report.json',
    f'{development_output_stem}_confusion_matrix.json',
    f'{development_output_stem}_confusion_matrix.png',
    'golden_parity.json',
    'holdout_manifest_validation.json',
    'training_requirements.txt',
]
for optional_name in [
    'physical_holdout_classification_report.json',
    'physical_holdout_confusion_matrix.json',
    'physical_holdout_confusion_matrix.png',
    'negative_guardrail.json',
]:
    if (OUTPUT_DIR / optional_name).is_file():
        artifact_names.append(optional_name)
artifact_hashes = {
    name: sha256_file(OUTPUT_DIR / name) for name in artifact_names
}

release_blockers = []
if not source_family_split_available:
    release_blockers.append(
        'Dataset development hanya memiliki satu source family per kelas; '
        'augmentation diagnostic tidak membuktikan generalisasi.'
    )
if holdout_summary is None:
    release_blockers.append('Independent physical holdout belum tersedia.')
if not holdout_manifest_valid:
    release_blockers.append(
        'Holdout manifest/provenance belum tersedia atau belum valid.'
    )
if negative_summary is None:
    release_blockers.append('Negative/non-Braille corpus belum tersedia.')
elif negative_summary['false_accept_count'] > 0:
    release_blockers.append('Model masih menerima negative sample di atas threshold.')
if not golden_parity_passed:
    release_blockers.append('Keras-TFLite golden parity belum lulus.')
release_blockers.append('Benchmark kamera, latency, FPS, soak, dan ground truth perangkat harus dilakukan di aplikasi.')

training_metadata = {
    'dataset': dataset_profile,
    'split': split_profile,
    'environment': environment,
    'preprocessing': preprocessing_contract,
    'class_indices': train_generator.class_indices,
    'model_parameter_count': model.count_params(),
    'selection': {
        'train_sample_count': int(train_generator.samples),
        'diagnostic_sample_count': int(tune_generator.samples),
        'best_epoch': best_epoch,
        'checkpoint': selection_checkpoint_path.name,
        'metric_role': development_evaluation_role,
    },
    'final_refit': final_refit_summary,
    'artifact_hashes': artifact_hashes,
    'caveat': split_caveat,
}
evaluation_report = {
    'release_status': 'blocked',
    'development_evaluation': development_summary,
    'physical_holdout': holdout_summary,
    'holdout_manifest_present': holdout_manifest_present,
    'holdout_manifest_validation': holdout_manifest_validation,
    'negative_guardrail': negative_summary,
    'keras_tflite_golden_parity': {
        key: value for key, value in golden_parity.items() if key != 'records'
    },
    'release_blockers': release_blockers,
}
save_json(OUTPUT_DIR / 'environment.json', environment)
save_json(OUTPUT_DIR / 'training_metadata.json', training_metadata)
save_json(OUTPUT_DIR / 'evaluation_report.json', evaluation_report)
package_hashes = dict(artifact_hashes)
for report_name in [
    'environment.json', 'training_metadata.json', 'evaluation_report.json'
]:
    package_hashes[report_name] = sha256_file(OUTPUT_DIR / report_name)
save_json(OUTPUT_DIR / 'artifact_hashes.json', package_hashes)

archive_base = WORKING_ROOT / 'braille_training_outputs'
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_DIR)
print(json.dumps({
    'release_status': evaluation_report['release_status'],
    'development_evaluation': development_summary,
    'release_blockers': release_blockers,
    'output_directory': str(OUTPUT_DIR),
    'archive': archive_path,
}, indent=2))

## Takeaways

1. Jalankan Save Version lalu Save & Run All di Kaggle.
2. Unduh braille_training_outputs.zip dari tab Output.
3. Jangan menyalin kandidat TFLite ke aplikasi sebelum evaluation_report.json, corpus fisik, dan benchmark perangkat ditinjau.
4. Jika split strategy adalah augmentation_diagnostic_fallback, accuracy hanya diagnostic dan tidak boleh dipakai sebagai klaim akurasi produk.
5. Status model aplikasi tetap unverified_candidate sampai seluruh evidence gate selesai.